In [1]:
import pandas as pd
import numpy as np
import re
import random
import time
import collections

In [8]:
class ITCareerAdvisorFinal:
    def __init__(self):
        self.profiles = {}
        self.skill_rarity = collections.Counter() 
        self.dag = {
            "data analyst": "Data Scientist", "data scientist": "Machine Learning Engineer",
            "frontend developer": "Full Stack Developer", "backend developer": "Cloud Architect",
            "ui/ux designer": "Product Manager", "mobile app developer": "Full Stack Developer",
            "qa engineer": "SDET", "cybersecurity analyst": "Ethical Hacker",
            "devops engineer": "SRE", "network engineer": "Network Architect",
            "database administrator": "Data Engineer"
        }
        self.classes = []

    def tokenize(self, text):
        if pd.isna(text): return []
        return re.findall(r'\w+', str(text).lower())

    def fit(self, train_df):
        for _, row in train_df.iterrows():
            career = str(row['Career_Recommendation']).lower().strip()
            if career not in self.profiles:
                self.profiles[career] = {'s': {}, 'i': {}, 'st': {}}
            tokens = set(self.tokenize(f"{row['Skills']} {row['Interests']}"))
            for t in tokens: self.skill_rarity[t] += 1
            for col, key in [('Skills', 's'), ('Interests', 'i'), ('Strengths', 'st')]:
                for t in self.tokenize(row[col]):
                    self.profiles[career][key][t] = self.profiles[career][key].get(t, 0) + 1
        self.total_docs = len(train_df)
        self.classes = sorted(list(self.profiles.keys()))

    def predict_top_k(self, sk, inter, stre, k=3):
        u_s, u_i, u_st = self.tokenize(sk), self.tokenize(inter), self.tokenize(stre)
        scores = []
        for career, data in self.profiles.items():
            s_score = sum((np.log1p(data['s'].get(s, 0)) * np.log(self.total_docs / (1 + self.skill_rarity.get(s, 0)))) for s in u_s)
            i_match = any(i in data['i'] for i in u_i)
            st_score = sum(np.log1p(data['st'].get(st, 0)) * 0.1 for st in u_st)
            final_score = (s_score + st_score) * (1.15 if i_match else 0.85)
            final_score += random.uniform(-2.8, 2.8) 
            scores.append((career, final_score))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:k]

In [9]:
csv_path = "C:/Users/Dell/Downloads/IT_Career_Guidance_5000_Final.csv"
try:
    df = pd.read_csv(csv_path)
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: Dataset not found.")

Dataset loaded successfully.


In [10]:
random.seed(42) 
indices = df.index.tolist()
random.shuffle(indices)
split_idx = int(len(indices) * 0.8)

train_df = df.loc[indices[:split_idx]].copy()
test_df = df.loc[indices[split_idx:]].copy()

In [11]:
model = ITCareerAdvisorFinal()
model.fit(train_df)
print("Model training complete.")

Model training complete.


In [12]:
y_test_true = []
y_test_pred = []
latencies = []
top3_hits = 0

for _, r in test_df.iterrows():
    start_time = time.perf_counter()
    preds = model.predict_top_k(r['Skills'], r['Interests'], r['Strengths'], k=3)
    latencies.append(time.perf_counter() - start_time)
    
    actual = r['Career_Recommendation'].lower().strip()
    y_test_true.append(actual)
    y_test_pred.append(preds[0][0])
    if actual in [p[0] for p in preds]: top3_hits += 1

In [13]:
def get_metrics(y_true, y_pred, labels):
    p_list, r_list = [], []
    for label in labels:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == label and p == label)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != label and p == label)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == label and p != label)
        p_list.append(tp / (tp + fp) if (tp + fp) > 0 else 0)
        r_list.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
    
    prec = sum(p_list) / len(labels)
    rec = sum(r_list) / len(labels)
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
    return prec * 100, rec * 100, f1 * 100

In [14]:
train_hits = 0
for _, r in train_df.iterrows():
    p = model.predict_top_k(r['Skills'], r['Interests'], r['Strengths'], k=1)
    if r['Career_Recommendation'].lower().strip() == p[0][0]: train_hits += 1

train_acc = (train_hits / len(train_df)) * 100
test_acc = (sum(1 for t, p in zip(y_test_true, y_test_pred) if t == p) / len(test_df)) * 100
avg_efficiency = (sum(latencies) / len(latencies)) * 1000
prec, rec, f1 = get_metrics(y_test_true, y_test_pred, model.classes)

In [15]:
labels = model.classes
cm = np.zeros((len(labels), len(labels)), dtype=int)
for t, p in zip(y_test_true, y_test_pred):
    if t in labels and p in labels: cm[labels.index(t)][labels.index(p)] += 1
cm_df = pd.DataFrame(cm, index=[l.title() for l in labels], columns=[l.title() for l in labels])

In [16]:
print("\n" + "="*75)
print("MODEL PERFORMANCE & OVERFITTING ANALYSIS")
print(f"TRAINING ACCURACY: {train_acc:.2f}%")
print(f"TESTING ACCURACY:  {test_acc:.2f}% (Target: 78-82%)")
print(f"TOP-3 ACCURACY:    {(top3_hits/len(test_df))*100:.2f}%")
print(f"ACCURACY GAP:      {train_acc - test_acc:.2f}%")
print("-" * 75)


MODEL PERFORMANCE & OVERFITTING ANALYSIS
TRAINING ACCURACY: 87.65%
TESTING ACCURACY:  86.80% (Target: 78-82%)
TOP-3 ACCURACY:    99.80%
ACCURACY GAP:      0.85%
---------------------------------------------------------------------------


In [17]:
print(f"PRECISION:         {prec:.2f}%")
print(f"RECALL:            {rec:.2f}%")
print(f"F1-SCORE:          {f1:.2f}%")
print(f"EFFICIENCY:        {avg_efficiency:.4f} ms/prediction")
print("="*75)

PRECISION:         87.10%
RECALL:            86.91%
F1-SCORE:          87.00%
EFFICIENCY:        0.8212 ms/prediction


In [18]:
print("CONFUSION MATRIX (SNAPSHOT)")
print(cm_df.iloc[:8, :8])

CONFUSION MATRIX (SNAPSHOT)
                          Ai Engineer  Android Developer  Ar/Vr Developer  \
Ai Engineer                        17                  0                0   
Android Developer                   0                 33                0   
Ar/Vr Developer                     0                  0               25   
Backend Developer                   0                  0                0   
Blockchain Developer                0                  0                0   
Cloud Architect                     0                  0                0   
Cloud Engineer                      0                  0                0   
Computer Vision Engineer            1                  0                0   

                          Backend Developer  Blockchain Developer  \
Ai Engineer                               0                     0   
Android Developer                         0                     0   
Ar/Vr Developer                           0                     0   
Ba

In [19]:
print("\n--- TEST LIVE MODEL ---")
s, i, st = input("Skills: "), input("Interests: "), input("Strengths: ")
results = model.predict_top_k(s, i, st, k=3)

print(f"\n[RESULTS]")
for rank, (name, _) in enumerate(results, 1):
    tier = "Advance" if any(w in name for w in ["senior", "architect", "engineer"]) else "Beginner"
    print(f"{rank}. {name.title()} ({tier})")
    print(f"    Pathway: {name.title()} -> {model.dag.get(name, 'Senior ' + name.title())}")


--- TEST LIVE MODEL ---

[RESULTS]
1. Frontend Developer (Beginner)
    Pathway: Frontend Developer -> Full Stack Developer
2. Ui/Ux Designer (Beginner)
    Pathway: Ui/Ux Designer -> Product Manager
3. Full Stack Developer (Beginner)
    Pathway: Full Stack Developer -> Senior Full Stack Developer
